## Initial baseline model

In [3]:
# Imports
import random
import os
import json
import re
from pathlib import Path
import tqdm
import cv2
from scipy import ndimage

import pandas as pd
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Rule-based extraction pipeline + evaluation
import random
try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except Exception:
    TESSERACT_AVAILABLE = False

# Configuration
DATA_DIR = Path("../finalproject_data/batch_1")  
CSV_FILES = [
    DATA_DIR / "batch1_1.csv",
    DATA_DIR / "batch1_2.csv",
    DATA_DIR / "batch1_3.csv"
]
# sample size
N_SAMPLE = 100
RANDOM_SEED = 42
USE_TESSERACT = True   # If USE_TESSERACT True but pytesseract not available, will fall back to CSV OCRed Text

In [ ]:
# Helper (same as above): enforce types
def enforce_invoice_dtypes(df):
    import numpy as np
    import pandas as pd

    text_cols = ["client_name", "seller_name", "invoice_number"]
    date_cols = ["invoice_date", "due_date"]

    df = df.copy()

    # TEXT FIELDS
    for col in text_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace("", np.nan)
            .replace("nan", np.nan)
        )

    # DATE FIELDS
    for col in date_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace("", np.nan)
            .replace("nan", np.nan)
        )
        df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in ["tax", "total_amount"]:
        # ensure column exists
        if col not in df.columns:
            df[col] = np.nan
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.strip()
            .replace("", np.nan)
            .replace("nan", np.nan)
        )

    # compute tax numeric value when tax is percent string like "10%"
    def compute_tax_value(tax_val, total_val):
        if pd.isna(tax_val):
            return np.nan

        # Percentage case (ends with %)
        if isinstance(tax_val, str) and tax_val.strip().endswith("%"):
            try:
                rate = float(tax_val.strip().replace("%", "")) / 100.0
                total_num = float(total_val)
                subtotal = total_num / (1.0 + rate)
                tax_amount = total_num - subtotal
                return round(tax_amount, 2)
            except Exception:
                return np.nan

        # Numeric case (already normalized string)
        try:
            return float(tax_val)
        except Exception:
            return np.nan

    # Convert total_amount to numeric (coerce)
    df["total_amount"] = pd.to_numeric(df["total_amount"], errors="coerce")

    # compute tax numeric
    if "tax" in df.columns:
        df["tax"] = df.apply(lambda r: compute_tax_value(r["tax"], r["total_amount"]), axis=1)
    else:
        df["tax"] = np.nan

    return df

# Load CSVs, parse JSON, extract fields
def load_and_flatten(csv_paths):
    dfs = []
    for p in csv_paths:
        if not p.exists():
            print("Missing CSV:", p)
            continue
        tmp = pd.read_csv(p)
        tmp["batch_csv"] = p.name
        # Parse JSON safely
        def parse_js(x):
            try:
                return json.loads(x)
            except Exception:
                return {}
        tmp["parsed_json"] = tmp["Json Data"].apply(parse_js)
        dfs.append(tmp)
    if not dfs:
        raise FileNotFoundError("No CSVs loaded. Check DATA_DIR and CSV_FILES")
    df = pd.concat(dfs, ignore_index=True)
    # Extract canonical fields from parsed_json
    def extract_fields(js):
        invoice = js.get("invoice", {}) if isinstance(js, dict) else {}
        subtotal = js.get("subtotal", {}) if isinstance(js, dict) else {}
        return {
            "client_name": invoice.get("client_name"),
            "seller_name": invoice.get("seller_name"),
            "invoice_number": invoice.get("invoice_number"),
            "invoice_date": invoice.get("invoice_date"),
            "due_date": invoice.get("due_date"),
            "tax": subtotal.get("tax"),
            "total_amount": subtotal.get("total")
        }
    extracts = df["parsed_json"].apply(lambda js: pd.Series(extract_fields(js)))
    df = pd.concat([df, extracts], axis=1)
    return df

df_all = load_and_flatten(CSV_FILES)
print("Loaded rows:", len(df_all))

# enforce dtypes (with tax % handling)
df_all = enforce_invoice_dtypes(df_all)

Loaded rows: 1414
